<a href="https://colab.research.google.com/github/takeuchi6459/Physical-AI/blob/main/02_llm_plans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03a LLMでプランJSONを作る
日本語の指示 → Qwen → `physical-ai-final/plans/<名前>.json` をDriveに保存。

In [ ]:
BASE = "/content/drive/MyDrive/physical-ai-final"
PLAN_DIR = BASE + "/plans"
LLM_ID = "Qwen/Qwen2.5-3B-Instruct"

from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(PLAN_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch, json, re
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float16, device_map={"": 0})

SYSTEM = """あなたは二足歩行ロボットの動作プランナーです。
ユーザーの指示を、次のJSON配列だけで出力してください。説明文は禁止です。
使えるaction:
- {"action":"forward","distance":メートル}
- {"action":"turn","direction":"left"か"right","angle_deg":度(正の数のみ)}
- {"action":"stop"}
Uターンは180度、「曲がる」「向く」で角度の指定がなければ90度です。
例:
指示: 3メートル進んで右に曲がって2メートル進んで
出力: [{"action":"forward","distance":3},{"action":"turn","direction":"right","angle_deg":90},{"action":"forward","distance":2}]
指示: 左を向いて
出力: [{"action":"turn","direction":"left","angle_deg":90}]
指示: 右に45度回って
出力: [{"action":"turn","direction":"right","angle_deg":45}]
指示: Uターンして1メートル戻って
出力: [{"action":"turn","direction":"left","angle_deg":180},{"action":"forward","distance":1}]"""

def plan(text):
    msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":f"指示: {text}\n出力:"}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inputs = tok(prompt, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=200, do_sample=False)
    s = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"\[.*\]", s, re.S)
    try:
        return json.loads(m.group(0)) if m else []
    except json.JSONDecodeError:
        return []

def sanitize(steps):                      # 安全層：範囲外・未知のactionを除去
    ok = []
    for st in steps[:10]:
        a = st.get("action")
        if a == "forward":
            ok.append({"action":"forward","distance":float(np.clip(st.get("distance",0),0,5))})
        elif a == "turn":
            ang = float(np.clip(abs(st.get("angle_deg", 90)), 0, 180))
            sign = 1 if st.get("direction") == "left" else -1   # 左=+, 右=−
            ok.append({"action":"turn","angle_deg": sign * ang})
        elif a == "stop":
            ok.append({"action":"stop"})
    return ok

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 指示を書いてJSON化

In [ ]:
INSTRUCTIONS = {
    "demo1_L":       "4メートルまっすぐ行って、左に曲がって、2メートル進んで",
    "demo2_U":       "右に曲がって3メートル歩いて、Uターンして戻ってきて",
    "demo3_square":  "2メートル進んで左に曲がるのを4回繰り返して",
    "demo4_zigzag":  "左に曲がって2メートル、そのあと右に曲がって2メートル",
    "demo5_short":   "ちょっと前に進んで",
}

for tag, text in INSTRUCTIONS.items():
    raw = plan(text)
    steps = sanitize(raw)
    print(f"[{tag}] {text}")
    print("  LLM出力:", json.dumps(raw, ensure_ascii=False))
    print("  安全層通過後:", json.dumps(steps, ensure_ascii=False))
    json.dump({"instruction": text, "plan": steps},
              open(f"{PLAN_DIR}/{tag}.json", "w"), ensure_ascii=False, indent=1)
    print("  saved:", f"{PLAN_DIR}/{tag}.json\n")

[demo1_L] 4メートルまっすぐ行って、左に曲がって、2メートル進んで
  LLM出力: [{"action": "forward", "distance": 4}, {"action": "turn", "direction": "left", "angle_deg": 90}, {"action": "forward", "distance": 2}]
  安全層通過後: [{"action": "forward", "distance": 4.0}, {"action": "turn", "angle_deg": 90.0}, {"action": "forward", "distance": 2.0}]
  saved: /content/drive/MyDrive/physical-ai-final/plans/demo1_L.json

[demo2_U] 右に曲がって3メートル歩いて、Uターンして戻ってきて
  LLM出力: [{"action": "turn", "direction": "right", "angle_deg": 90}, {"action": "forward", "distance": 3}, {"action": "turn", "direction": "left", "angle_deg": 180}]
  安全層通過後: [{"action": "turn", "angle_deg": -90.0}, {"action": "forward", "distance": 3.0}, {"action": "turn", "angle_deg": 180.0}]
  saved: /content/drive/MyDrive/physical-ai-final/plans/demo2_U.json

[demo3_square] 2メートル進んで左に曲がるのを4回繰り返して
  LLM出力: [{"action": "forward", "distance": 2}, {"action": "turn", "direction": "left", "angle_deg": 90}, {"action": "forward", "distance": 2}, {"action": "turn", "direction":